# Phase 3 — Retrieval Evaluation
**Multimodal RAG for E-commerce**

This notebook produces the quantitative evaluation the project spec requires: `Recall@1`, `Recall@5`, `Recall@10` across multiple query types.

The classic challenge with RAG eval on an unlabeled product catalog is that we don't have ground-truth (query, relevant_product) pairs. We work around it with three eval strategies:

1. **Self-retrieval (text)**: use each product's own description as the query — the correct answer is the product itself. This is the easiest test; if recall isn't near-perfect here, something's broken in the index.
2. **Self-retrieval (image)**: same idea using each product's image. Tests that the image embedding correctly identifies the product.
3. **Partial-text retrieval**: use just the product title + brand as the query, evaluate against the full-description index. This simulates real user queries where they know less than the full description.
4. **Cross-modal retrieval**: query the *image* index with a *text* query (and vice versa). This tests CLIP's shared-space alignment specifically.

Save the results to a CSV at the end — these go straight into your research report.

## 1. Setup

In [ ]:
# !pip install -q transformers torch chromadb pandas numpy tqdm

import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPModel, CLIPProcessor
import chromadb

DATA_DIR = Path("./prepared_data")
CHROMA_DIR = Path("./chroma_db")
RESULTS_DIR = Path("./eval_results")
RESULTS_DIR.mkdir(exist_ok=True)

MODEL_NAME = "openai/clip-vit-base-patch32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Eval config
N_EVAL = 500              # sample size for eval (use full set if you have time)
K_VALUES = [1, 5, 10]     # spec asks for Recall@1, @5, @10
RANDOM_SEED = 42

print(f"Device: {DEVICE}")

## 2. Load model, data, and collections

In [ ]:
model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

df = pd.read_parquet(DATA_DIR / "products_indexed.parquet")
print(f"Loaded {len(df):,} indexed products")

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
text_collection = client.get_collection("products_text")
image_collection = client.get_collection("products_image")
print(f"Collections: text={text_collection.count()}, image={image_collection.count()}")

## 3. Helper functions

In [ ]:
def l2_normalize(x):
    norms = np.linalg.norm(x, axis=-1, keepdims=True)
    norms = np.where(norms == 0, 1e-12, norms)
    return x / norms

@torch.no_grad()
def encode_text(texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True, max_length=77)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    feats = model.get_text_features(**inputs).cpu().numpy()
    return l2_normalize(feats)

@torch.no_grad()
def encode_image(image_paths):
    imgs = [Image.open(p).convert("RGB") for p in image_paths]
    inputs = processor(images=imgs, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    feats = model.get_image_features(**inputs).cpu().numpy()
    return l2_normalize(feats)

def recall_at_k(retrieved_ids, gold_id, k):
    """1 if gold_id appears in the top-k retrieved IDs, else 0."""
    return int(gold_id in retrieved_ids[:k])

def evaluate(query_embeddings, gold_ids, collection, max_k=10):
    """Run all queries against `collection`, return DataFrame of per-query results."""
    results = []
    BATCH = 32
    for i in range(0, len(query_embeddings), BATCH):
        batch_embs = query_embeddings[i:i+BATCH]
        batch_gold = gold_ids[i:i+BATCH]
        res = collection.query(
            query_embeddings=batch_embs.tolist(),
            n_results=max_k,
            include=["distances"],
        )
        for j, gold in enumerate(batch_gold):
            retrieved = res["ids"][j]
            row = {"gold_id": gold}
            for k in K_VALUES:
                row[f"recall@{k}"] = recall_at_k(retrieved, gold, k)
            results.append(row)
    return pd.DataFrame(results)

## 4. Build the evaluation set

We sample a subset of products for evaluation (the full set works too but takes longer).

In [ ]:
eval_df = df.sample(min(N_EVAL, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Eval set: {len(eval_df):,} products")
print(f"Categories represented: {eval_df['top_category'].nunique()}")

## 5. Evaluation #1: Text self-retrieval

Use the indexed description (`desc_standard`) as the query. Recall@1 should be very high (near 1.0) — this is essentially asking "can the index find the exact thing I stored?"

If Recall@1 is < 0.95, you have duplicates or near-duplicates in the catalog (very similar products competing for the top spot). That's a finding to mention in your report, not a bug.

In [ ]:
queries_text = eval_df["desc_standard"].tolist()
gold_ids = eval_df["product_id"].tolist()

print("Encoding text queries...")
q_embs = encode_text(queries_text)

print("Querying text collection...")
results_t2t = evaluate(q_embs, gold_ids, text_collection)
results_t2t["query_type"] = "text→text (self)"

print("\nText self-retrieval results:")
for k in K_VALUES:
    print(f"  Recall@{k}: {results_t2t[f'recall@{k}'].mean():.4f}")

## 6. Evaluation #2: Image self-retrieval

Use each product's own image as the query against the image index. Same logic as above — Recall@1 should be near 1.0.

In [ ]:
queries_img_paths = eval_df["local_image_path"].tolist()
gold_ids = eval_df["product_id"].tolist()

print("Encoding image queries...")
BATCH = 64
img_embs = []
for i in tqdm(range(0, len(queries_img_paths), BATCH)):
    batch = queries_img_paths[i:i+BATCH]
    img_embs.append(encode_image(batch))
img_embs = np.vstack(img_embs)

print("Querying image collection...")
results_i2i = evaluate(img_embs, gold_ids, image_collection)
results_i2i["query_type"] = "image→image (self)"

print("\nImage self-retrieval results:")
for k in K_VALUES:
    print(f"  Recall@{k}: {results_i2i[f'recall@{k}'].mean():.4f}")

## 7. Evaluation #3: Partial-text retrieval

Realistic user queries don't contain the full description. They typically have just the product name, brand, or a short attribute phrase. This is the harder, more realistic test.

In [ ]:
partial_queries = (eval_df["Product Name"].fillna("") + " " + eval_df["Brand Name"].fillna("")).str.strip().tolist()
gold_ids = eval_df["product_id"].tolist()

print("Encoding partial queries...")
q_embs = encode_text(partial_queries)

print("Querying text collection...")
results_partial = evaluate(q_embs, gold_ids, text_collection)
results_partial["query_type"] = "partial-text→text"

print("\nPartial-text retrieval results:")
for k in K_VALUES:
    print(f"  Recall@{k}: {results_partial[f'recall@{k}'].mean():.4f}")

## 8. Evaluation #4: Cross-modal retrieval (image → text and text → image)

Use the image embedding to query the text index, and vice versa. This is the most important multimodal test — it directly measures whether CLIP's text and image spaces are aligned in your indexed catalog.

In [ ]:
# Image queries against text index
print("Querying text index with image embeddings...")
results_i2t = evaluate(img_embs, gold_ids, text_collection)
results_i2t["query_type"] = "image→text"

# Text queries against image index
print("Querying image index with text embeddings...")
q_text_embs = encode_text(eval_df["desc_standard"].tolist())
results_t2i = evaluate(q_text_embs, gold_ids, image_collection)
results_t2i["query_type"] = "text→image"

print("\nCross-modal retrieval:")
print("  Image→Text:")
for k in K_VALUES:
    print(f"    Recall@{k}: {results_i2t[f'recall@{k}'].mean():.4f}")
print("  Text→Image:")
for k in K_VALUES:
    print(f"    Recall@{k}: {results_t2i[f'recall@{k}'].mean():.4f}")

## 9. Summary table

In [ ]:
all_results = pd.concat([results_t2t, results_i2i, results_partial, results_i2t, results_t2i], ignore_index=True)

summary = all_results.groupby("query_type")[[f"recall@{k}" for k in K_VALUES]].mean().round(4)
summary = summary.reindex([
    "text→text (self)",
    "image→image (self)",
    "partial-text→text",
    "image→text",
    "text→image",
])
print(summary)

# Save for the report
summary.to_csv(RESULTS_DIR / "recall_summary.csv")
all_results.to_csv(RESULTS_DIR / "per_query_results.csv", index=False)
print(f"\nSaved: {RESULTS_DIR / 'recall_summary.csv'}")
print(f"Saved: {RESULTS_DIR / 'per_query_results.csv'}")

## 10. Per-category breakdown

In [ ]:
# Which categories are easier/harder to retrieve?
all_results["category"] = all_results["gold_id"].map(
    df.set_index("product_id")["top_category"].to_dict()
)

cat_breakdown = (
    all_results[all_results["query_type"] == "partial-text→text"]
    .groupby("category")[[f"recall@{k}" for k in K_VALUES]]
    .agg(["mean", "count"])
    .round(3)
)

# Sort by Recall@5 mean
cat_breakdown_sorted = cat_breakdown.sort_values(("recall@5", "mean"), ascending=False)
print("Top 10 best-performing categories (partial-text retrieval):")
print(cat_breakdown_sorted.head(10))

print("\nBottom 10 worst-performing categories:")
print(cat_breakdown_sorted.tail(10))

cat_breakdown.to_csv(RESULTS_DIR / "recall_by_category.csv")

## 11. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
summary.plot(kind="bar", ax=ax)
ax.set_ylabel("Recall")
ax.set_title("Retrieval performance by query type")
ax.set_ylim(0, 1.05)
ax.legend(title="k")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "recall_chart.png", dpi=120)
plt.show()

## What's next — Phase 4 & 5

You have the numbers. Plug them into the report's evaluation section. Then move to:
- **`../app/rag_chain.py`** — the RAG retrieval + LLM call logic
- **`../app/app.py`** — the Streamlit chatbot UI

These tie everything together: user query → CLIP encode → ChromaDB retrieve → LLM generates grounded answer.